In [40]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages

In [41]:
model = ChatOllama(model="llama3")

In [ ]:
'''
1. The BaseMessage field contains a list of chat messages like as below:
 [
    HumanMessage(content="Hello"),
    AIMessage(content="Hi"),
]

2. add_messages => 
`add_messages` is a LangGraph reducer used to append new chat messages to the existing conversation history instead of replacing them.
When a node returns new messages, append them to the existing messages instead of replacing them.
'''
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [43]:
def chat_node(state: ChatState) :
    # take user query from state
    message = state['messages']

    # send to llm
    response = model.invoke(message)

    # response store state
    return {'messages': [response]}

In [44]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

# workflow = graph.compile()

In [ ]:
# initial_state = {'messages': [HumanMessage(content="What is the capital of France?")]}
# workflow.invoke(initial_state)  


{'messages': [HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}, id='50b83103-9afc-43c3-89e0-9f0f35c10a0d'),
  AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2026-06-17T09:40:12.2009239Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 19020371300, 'load_duration': 15883259900, 'prompt_eval_count': 17, 'prompt_eval_duration': 1148587000, 'eval_count': 8, 'eval_duration': 1974013000}, id='lc_run--019ed4f3-8aeb-72e0-914f-74e86d63ff23-0', tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
# initial_state = {'messages': [HumanMessage(content="What is the capital of India?")]}
# workflow.invoke(initial_state)  

{'messages': [HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='c1346558-5f70-4273-ae75-23d392d7f2f9'),
  AIMessage(content='The capital of India is New Delhi.', additional_kwargs={}, response_metadata={'model': 'llama3', 'created_at': '2026-06-17T09:40:17.6685775Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 3435451800, 'load_duration': 285469300, 'prompt_eval_count': 17, 'prompt_eval_duration': 918947000, 'eval_count': 9, 'eval_duration': 2225235000}, id='lc_run--019ed4f3-dd3d-7300-b689-ceeccd4f3029-0', tool_calls=[], invalid_tool_calls=[])]}

In [ ]:
# this code is not remembering the previous conversation, every time we are giving new input to the model, it is not aware of the previous conversation. To make it aware of the previous conversation we need to append the new messages to the existing messages instead of replacing them.
workflow = graph.compile()
while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    initial_state = {'messages': [HumanMessage(content=user_input)]}
    response = workflow.invoke(initial_state) 
    print('User:', user_input) 
    print("AI:", response['messages'][-1].content)

User: hi\
AI: Hi! It's nice to meet you. Is there something I can help you with, or would you like to chat?
User: my name is swati
AI: Nice to meet you, Swati! That's a lovely name. What brings you here today? Would you like to chat about something specific or just explore and see where the conversation takes us?
User: what is my name
AI: I apologize, but I don't know your name. You are interacting with me as a chatbot or AI assistant, and we haven't had any prior conversation or exchange of information that would allow me to know your name. Each time you interact with me, it's a new conversation and I don't retain any information from previous conversations. If you'd like to share your name with me, I'm happy to learn more about you!


In [ ]:
# persistence concept saves data 
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer)

config = {"configurable": {"thread_id": "1"}}

while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    initial_state = {'messages': [HumanMessage(content=user_input)]}
    response = workflow.invoke(initial_state, config=config) 
    print('User:', user_input) 
    print("AI:", response['messages'][-1].content)

User: hi
AI: Hi! It's nice to meet you. Is there something I can help you with or would you like to chat?
User: my name is mahesh
AI: Nice to meet you, Mahesh! What brings you here today? Do you have a particular question or topic you'd like to discuss, or are you just looking for some friendly conversation? I'm all ears!
User: what is my name
AI: I know this one! Your name is Mahesh, right?
User: who is founder of insta?
AI: The founder of Instagram is Kevin Systrom and Mike Krieger. They launched the app in 2010, and it was later acquired by Facebook (owned by Mark Zuckerberg) in 2012.


In [ ]:
# added streaming concept which print responce as it is generated by the model instead of waiting for the complete response to be generated and then printing it.
# streaming token-by-token.
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer)

config = {"configurable": {"thread_id": "1"}}

while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    initial_state = {'messages': [HumanMessage(content=user_input)]}
    print('User:', user_input)
    print("AI: ", end="", flush=True)
    for chunk, metadata  in workflow.stream(initial_state, config=config, stream_mode="messages"):
        #chunk is dict with keys: content, node_id, state, is_final
        print(chunk.content, end="", flush=True)
    print("\n")
    # response = workflow.invoke(initial_state, config=config) 
    # print("AI:", response['messages'][-1].content)

User: My name is Mahesh Ghatamnneni
AI: Nice to meet you, Mahesh! It's great to have a conversation with someone from India, especially since I'm learning about different cultures and languages every day. What brings you here today? Do you have any specific questions or topics you'd like to discuss?

User: I am south indian person
AI: As a South Indian person, you're part of a rich and diverse cultural heritage! South India has a unique identity shaped by its history, languages, cuisine, and traditions. Which region are you from - Tamil Nadu, Andhra Pradesh, Karnataka, Kerala, or Telangana?

Do you have any favorite dishes or snacks that you'd like to share? Idlis, dosas, vadas, and sambar are some of the most popular South Indian staples!

What do you think is the most distinctive aspect of South Indian culture? Is it the vibrant festivals, the colorful traditional clothing, or something else entirely?

User: i am from Telangana
AI: Wonderful! Telangana is a beautiful state with a ric